In [2]:
import pandas as pd
import json

# 1. Đọc file
df = pd.read_csv('training_data_final_merged.csv')

tasks = []
for index, row in df.iterrows():
    if pd.notna(row['input_text']):
        text_content = str(row['input_text'])
        
        # Tách Context và Comment dựa trên '</s>'
        if '</s>' in text_content:
            parts = text_content.split('</s>')
            context_val = parts[0].strip()
            comment_val = " ".join(parts[1:]).strip() # Gộp lại nếu có nhiều hơn 1 dấu </s>
        else:
            # Trường hợp không có dấu phân cách
            context_val = "Không có ngữ cảnh"
            comment_val = text_content

        # Tạo task data
        task = {
            "data": {
                "context": context_val,
                "comment": comment_val
            }
        }

        # Xử lý Pre-label (Nhãn có sẵn)
        if pd.notna(row['label']):
            label_val = str(int(row['label']))
            task["predictions"] = [{
                "model_version": "v1_pre_labeled",
                "result": [{
                    "from_name": "label",
                    "to_name": "comment_text", # Link tới phần comment text
                    "type": "choices",
                    "value": { "choices": [label_val] }
                }]
            }]
        
        # Xử lý Note (nếu có)
        if 'note' in row and pd.notna(row['note']):
             task["predictions"][0]["result"].append({
                "from_name": "note",
                "to_name": "comment_text",
                "type": "textarea",
                "value": { "text": [str(row['note'])] }
            })

        tasks.append(task)

# 2. Xuất file JSON
with open('tasks_split_context.json', 'w', encoding='utf-8') as f:
    json.dump(tasks, f, ensure_ascii=False, indent=2)

print("Đã tạo file 'tasks_split_context.json'.")

Đã tạo file 'tasks_split_context.json'.
